In [0]:
%pip install plotly

In [0]:
df = spark.table("energy_weather.default.solar_pv_compare")

# Stundensumme Globalstrahlung vs. PV-Erzeugung je Stunde
from pyspark.sql import functions as F

agg = (df.groupBy("stunde_re")
         .agg(F.sum("FG_LBERG").alias("globalradiation_kWh_per_m2"),
              F.first("Photovoltaik_MWh").alias("pv_mwh"))
         .orderBy("stunde_re"))
display(agg)

Databricks visualization. Run in Databricks to view.

**Jahresvergleich: Globalstrahlung vs. PV-Erzeugung (Tagesbasis, Liniendiagramm)**

Für das Chart über das ganze Jahr wird auf Tagesebene aggregiert – Stundenwerte
wären in einer Jahresansicht unlesbar (insbesondere, da die Globalstrahlung Stundenweise 0 sein kann). Wichtig: Photovoltaik_MWh steht nach dem Station-Join je Stunde mehrfach (pro Station) im Gold-Datensatz; für die
PV-Summe darf es daher nur einmal pro Stunde gezählt werden (first), sonst
wird die Erzeugung um den Faktor der Stationsanzahl vervielfacht.

Das Chart wird mit Plotly gezeichnet – vorinstalliert in Databricks Runtime
(DBR). Es ist bewusst nicht fig.show() verwendet: das funktioniert in
Notebooks interaktiv nur teilweise zuverlässig (leeres Output, Embedded-App-Konflikt
mit display()); in Databricks zeigt stattdessen displayHTML(fig.to_html())
das Chart zuverlässig inline an (und auch fig.show() selbst nutzt intern HTML).
Erst nach toPandas() am Driver – die Tagesaggregate sind mit ~365 Zeilen klein
und unkritisch.

In [0]:
import plotly.graph_objects as go
from pyspark.sql import functions as F

df = spark.table("energy_weather.default.solar_pv_compare")

# --- 1) je Stunde: PV nur einmal zählen, Strahlung über Stationen summieren --
stunde = (df.groupBy("stunde_re")
             .agg(F.sum("FG_LBERG").alias("globalstrahlung_sum"),
                  F.first("Photovoltaik_MWh").alias("pv_mwh")))

# --- 2) auf Tagesebene aggregieren -----------------------------------------
tag_spark = (stunde.groupBy(F.date_trunc("day", "stunde_re").alias("tag"))
                   .agg(F.sum("globalstrahlung_sum").alias("globalstrahlung_tag"),
                        F.sum("pv_mwh").alias("pv_mwh_tag"))
                   .orderBy("tag"))

pdf = tag_spark.toPandas()          # ~365 Zeilen, unkritisch am Driver

# --- 3) Plotly-Linienchart -------------------------------------------------
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=pdf["tag"], y=pdf["globalstrahlung_tag"],
    mode="lines", name="Globalstrahlung (kWh/m², Stationssumme)",
    line=dict(color="#f2a541")))
fig.add_trace(go.Scatter(
    x=pdf["tag"], y=pdf["pv_mwh_tag"],
    mode="lines", name="PV-Erzeugung (MWh)",
    line=dict(color="#2a9d8f"), yaxis="y2"))   # zweite Achse: eigene Einheit

fig.update_layout(
    title="Globalstrahlung vs. Photovoltaik-Erzeugung – Tageswerte 2025",
    xaxis_title="Datum",
    yaxis=dict(title="Globalstrahlung (kWh/m²)"),
    yaxis2=dict(title="PV-Erzeugung (MWh)", overlaying="y", side="right"),
    legend=dict(orientation="h", y=1.12),
    hovermode="x unified",
    template="plotly_white",
    height=450,
)

displayHTML(fig.to_html())